In [1]:
# Choose a theme to your convenience
PLOTLY_THEME = "plotly_dark"
#PLOTLY_THEME = "plotly_white"


In [2]:
from dataclasses import dataclass, asdict

@dataclass
class SimulationParameters:
    re: float
    N: int
    steps: int
    sampling: int
    K: float
    dt: float
    pressure_iters: int
    precond_steps: int
    precond_dt: float
    full_solve_iters: int
    scheme: str
    seed: int

In [3]:
import configparser
def write_config(params):
    config = configparser.ConfigParser()
    config["Simulation"] = asdict(params)
    with open('simulation.ini', 'w') as configfile:
      config.write(configfile)

In [4]:
import subprocess,  os

def launch_c():
    subprocess.run(["cmake", "--build", "build"])
    
    with open(os.devnull, "w") as devnull:
        proc = subprocess.Popen(
            ["./build/cldc"],
            stdout=devnull,
            stderr=devnull
        )

In [5]:
import mmap
import struct
import time

from tqdm.notebook import tqdm

import numpy as np
import plot_run
import posix_ipc

def launch_run(params, graph_out=True):
    write_config(params)
    FLAT_ARRAY_SIZE = params.N * params.N * 3
    record_size = params.steps // params.sampling
    output_array = np.zeros((record_size, params.N, params.N, 3))
    
    shm = posix_ipc.SharedMemory("/sim_shm", posix_ipc.O_CREAT, size=8 + 8 * FLAT_ARRAY_SIZE)
    mm = mmap.mmap(shm.fd, shm.size)
    
    shm.close_fd()
    
    mm.write(struct.pack("Q", 0))
    
    write_idx = np.frombuffer(mm, dtype=np.uint64, count=1, offset=0)
    data = np.frombuffer(mm, dtype=np.float64, count=FLAT_ARRAY_SIZE, offset=8)
    if graph_out:
        plotter = plot_run.RunPlot(params.re, params.K, params.N, PLOTLY_THEME)
        plotter.prepare()
    last = 0
    try:
        pbar = tqdm(total=params.steps)
        pbar.set_description("Waiting for C simulation")
        launch_c()
        while True:
            w = int(write_idx[0])
            index = w // params.sampling
            if w > last:
                pbar.set_description("")
                pbar.update(params.sampling)
                last = w
                velocity = data[:params.N * params.N * 2].copy().reshape(params.N, params.N, 2)
                pressure = data[params.N * params.N * 2:params.N * params.N * 3].copy().reshape(params.N, params.N, 1)
                frame = np.concatenate([velocity, pressure], axis=-1)
                output_array[index] = frame
                if graph_out:
                    plotter.update(velocity, pressure, index)
            if index >= record_size - 1:
                break
            time.sleep(0.00001)
    except KeyboardInterrupt:
        print("User stopped run")
    print("Run done")
    # np.save("runs/out", output_array)
    return output_array


## Init storage

In [6]:
import os
with open(os.path.expanduser("~") + "/.keys") as f:
    for line in f:
        if line.startswith("export "):
            key, val = line.strip().split("=", 1)
            os.environ[key.replace("export ","")] = val


import boto3
import zarr
import s3fs

X = 128
T = 100
C_in = 4
C_out = 3

fs = s3fs.S3FileSystem(
    key=os.environ["AWS_ACCESS_KEY_ID"],
    secret=os.environ["AWS_SECRET_ACCESS_KEY"],
    client_kwargs={"endpoint_url": "http://localhost:8333"}
)

store_X = s3fs.S3Map(root="ldcdataset/ds2/X", s3=fs, check=False)
store_Y = s3fs.S3Map(root="ldcdataset/ds2/Y", s3=fs, check=False)

def init_bucket():
    s3 = boto3.client(
        "s3",
        endpoint_url="http://localhost:8333",
        aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
        aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
    )
    s3.create_bucket(Bucket="ldcdataset")


def init_dataset(): 
    zarr.open(store_X, mode="w", shape=(0, X, X, C_in), chunks=(1, X, X, C_in), dtype='f8')
    zarr.open(store_Y, mode="w", shape=(0, T, X, X, C_out), chunks=(1, X, X, T, C_out), dtype='f8')

# init_dataset()

dts_X = zarr.open(store_X, mode="a")
dts_Y = zarr.open(store_Y, mode="a")

In [7]:
dts_X.shape

(0, 128, 128, 4)

In [8]:

dt = 1e-3
sr = 50
prec = 150
steps = 10150
t_start = 0
t_end = 10
t_inter = int((t_end - t_start) / T / dt / sr)
tmid_idx = int(t_start / t_end * steps / sr)
tlast_idx = tmid_idx + T * t_inter

In [9]:
seeds = [i for i in range(100)]

param_set = [SimulationParameters(500, 128, steps, sr, 1.0, dt, 5, prec, 1e-5, 100, "RK4", seed * 100_000) for seed in seeds]

In [10]:


def param_to_entry(param):
    nu = 1.0 / param.re
    out = launch_run(param, False)
    # out of shape iter / sr, X, X, 3 (u v p)
    
    # data x : X, X, C with C: icu0, icu1, icp, nu
    
    data_x = np.concatenate([out[tmid_idx], (np.ones((X, X)) * nu)[..., None]], axis=-1)
    # data y : T, X, X, C with C: u0, u1, p with t=0 t0idx
    data_y = out[tmid_idx:tlast_idx:t_inter]
    dts_X.append(data_x[None, ...])
    dts_Y.append(data_y[None, ...])

In [11]:
for param in tqdm(param_set):
    param_to_entry(param)


  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/10150 [00:00<?, ?it/s]

[100%] Built target cldc
Run done


  0%|          | 0/10150 [00:00<?, ?it/s]

[100%] Built target cldc
Run done


...

[100%] Built target cldc
Run done


  0%|          | 0/10150 [00:00<?, ?it/s]

[100%] Built target cldc
Run done


  0%|          | 0/10150 [00:00<?, ?it/s]

[100%] Built target cldc
Run done
